# intervalNets speed test notebook

Benchmark `lpnorm` and `sobolev_norm` runtime on random 1D and 3D networks.
Use this notebook across versions to compare performance changes.


In [ ]:
import time
import torch
from torch import nn

from intervalnets import IntervalTensor, enable_interval_eval

torch.set_default_dtype(torch.float32)
enable_interval_eval()


In [ ]:
P_VAL = 2.0
ITERATIONS = 20
THETA = 0.5
SPLIT_TOPK = 2
BATCH_SIZE = 64

print(f"p={P_VAL}, iterations={ITERATIONS}, theta={THETA}, split_topk={SPLIT_TOPK}, batch_size={BATCH_SIZE}")


In [ ]:
def make_random_network(dim: int, width: int, seed: int) -> nn.Module:
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Linear(dim, width),
        nn.ReLU(),
        nn.Linear(width, width),
        nn.ReLU(),
        nn.Linear(width, 1),
    )
    with torch.no_grad():
        for param in model.parameters():
            nn.init.uniform_(param, a=-1.0, b=1.0)
    return model


def run_case(name: str, model: nn.Module, domain: IntervalTensor) -> dict:
    # Warmup pass to avoid one-time overhead skew in timing.
    _ = model.lpnorm(domain, p=P_VAL, iterations=1, theta=THETA, split_topk=SPLIT_TOPK, batch_size=BATCH_SIZE)

    t0 = time.perf_counter()
    lp_bounds = model.lpnorm(domain, p=P_VAL, iterations=ITERATIONS, theta=THETA, split_topk=SPLIT_TOPK, batch_size=BATCH_SIZE)
    lp_time = time.perf_counter() - t0

    t1 = time.perf_counter()
    w1p_bounds = model.sobolev_norm(domain, p=P_VAL, iterations=ITERATIONS, theta=THETA, split_topk=SPLIT_TOPK, batch_size=BATCH_SIZE)
    w1p_time = time.perf_counter() - t1

    result = {
        "name": name,
        "lp_time_sec": lp_time,
        "w1p_time_sec": w1p_time,
        "lp_bounds": (float(lp_bounds.lower), float(lp_bounds.upper)),
        "w1p_bounds": (float(w1p_bounds.lower), float(w1p_bounds.upper)),
    }
    return result


In [ ]:
# 1D random network
net_1d = make_random_network(dim=1, width=32, seed=123)
domain_1d = IntervalTensor.from_bounds([-1.0], [1.0])

# 3D random network
net_3d = make_random_network(dim=3, width=64, seed=456)
domain_3d = IntervalTensor.from_bounds([-1.0, -0.5, 0.0], [1.0, 1.5, 2.0])

results = [
    run_case("random_1d", net_1d, domain_1d),
    run_case("random_3d", net_3d, domain_3d),
]

for item in results:
    print("-" * 80)
    print(f"Case: {item['name']}")
    print(f"  Lp time   : {item['lp_time_sec']:.6f} s")
    print(f"  W1p time  : {item['w1p_time_sec']:.6f} s")
    print(f"  Lp bounds : {item['lp_bounds']}")
    print(f"  W1p bounds: {item['w1p_bounds']}")


## Notes

- Keep this notebook and run it on older versions to compare wall-clock runtime.
- For fair comparisons, keep hardware/environment, seeds, and settings fixed.
